In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os
from pathlib import Path
from datetime import datetime

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.features.features_v1 import *
from src.features.features_v2 import *
from src.pipeline.calculate_evs import *
from src.utils.helper_functions import *
from src.utils.team_info import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 4 teams with confirmed lineups


### Load Model

### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

if us_file is None:
    raise FileNotFoundError(f"No NBA_US file found for {today}")
if dfs_file is None:
    raise FileNotFoundError(f"No NBA_DFS file found for {today}")

s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
usData = pd.read_csv(us_file)
dfsData = pd.read_csv(dfs_file)

print(f"Loaded: {us_file.name}")
print(f"Loaded: {dfs_file.name}")
dfsData.head()

Loaded: NBA_US_20251209_151437.csv
Loaded: NBA_DFS_20251209_151330.csv


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Underdog,player_points,Paolo Banchero,Over,20.5,-137,2025-12-09,2025-12-09T23:11:51Z,2025-12-09 15:13:30
1,Underdog,player_points,Paolo Banchero,Under,20.5,-137,2025-12-09,2025-12-09T23:11:51Z,2025-12-09 15:13:30
2,Underdog,player_points,Bam Adebayo,Over,17.5,-137,2025-12-09,2025-12-09T23:11:51Z,2025-12-09 15:13:30
3,Underdog,player_points,Bam Adebayo,Under,17.5,-137,2025-12-09,2025-12-09T23:11:51Z,2025-12-09 15:13:30
4,Underdog,player_points,Jalen Suggs,Over,18.5,-137,2025-12-09,2025-12-09T23:11:51Z,2025-12-09 15:13:30


In [4]:
from src.features.feature_engine import FeatureEngine

engine = FeatureEngine({
    "min_model": "src/models/saved/min_model.pkl",
    "usg_model": "src/models/saved/usg_model.pkl",
    "fga_model": "src/models/saved/fga_model.pkl",
    "ngboost_model_wrapper": "src/models/saved/pts_model_wrapper.pkl"
})

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/nba_model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## For Post Analysis

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

singleBets = calculateSingleBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,                  
    max_player_appearances=1,  
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)
singleBets.to_csv(f'notebooks/exploration/old_evs/singleBets_UD_{current_date}.csv', index=False)
singleBets

Computing predictions for 30 players...
[MIN] No data found for Wendell Carter Jr
Found 29 valid players


,NAME,LINE,SIDE,PREDICTION,MODEL_PROB,IMPLIED_PROB,EDGE,BET_EDGE,ODDS,DECIMAL_ODDS,EV,EV_PERCENT,KELLY_QUARTER,TEAM,OPPONENT
4,Jaime Jaquez Jr.,10.5,over,20.78,0.970,0.49,10.28,0.480,100,2.000,0.9396,93.96,0.2349,MIA,ORL
27,Isaiah Joe,7.5,over,14.38,0.891,0.50,6.88,0.391,-105,1.952,0.7398,73.98,0.1942,OKC,PHX
8,Kel'el Ware,8.5,over,12.99,0.850,0.49,4.49,0.360,-102,1.980,0.6835,68.35,0.1743,MIA,ORL
24,Mark Williams,9.5,over,13.29,0.809,0.54,3.79,0.269,-122,1.820,0.4720,47.20,0.1439,PHX,OKC
7,Anthony Black,15.5,under,10.22,0.756,0.51,5.28,0.246,-110,1.909,0.4429,44.29,0.1218,ORL,MIA
18,Devin Booker,22.5,over,26.48,0.745,0.52,3.98,0.225,-115,1.870,0.3933,39.33,0.1131,PHX,OKC
9,Dru Smith,4.5,over,6.64,0.716,0.51,2.14,0.206,-110,1.909,0.3674,36.74,0.1010,MIA,ORL
21,Dillon Brooks,18.5,over,22.74,0.788,0.56,4.24,0.228,-137,1.730,0.3631,36.31,0.1244,PHX,OKC
2,Jalen Suggs,18.5,under,12.20,0.709,0.51,6.30,0.199,-110,1.909,0.3544,35.44,0.0975,ORL,MIA
3,Tyler Herro,22.5,over,24.51,0.642,0.49,2.01,0.152,100,2.000,0.2833,28.33,0.0708,MIA,ORL


In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=100,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)


# prizepicksPairs.to_csv(f'notebooks/exploration/old_evs/prizepicksPairs_{current_date}.csv', index=False)
prizepicksPairs.to_csv(f'notebooks/exploration/old_evs/underdogPairs_{current_date}.csv', index=False)
prizepicksPairs

Computing predictions for 30 players...
[MIN] No data found for Wendell Carter Jr
Found 29 valid players
Generated 274 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,SIDE 1,SIDE 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,IMPLIED_PROB 1,IMPLIED_PROB 2,PARLAY_PROB,PARLAY_IMPLIED_PROB,PARLAY_EDGE,EDGE 1,EDGE 2,ODDS 1,ODDS 2,PARLAY_ODDS,PARLAY_DECIMAL,EV,EV_PERCENT,KELLY_QUARTER
93,Jaime Jaquez Jr.,Isaiah Joe,10.5,7.5,over,over,20.78,14.38,0.970,0.891,0.49,0.50,0.864,0.245,0.619,10.28,6.88,100,-105,290,3.90,2.3704,237.04,0.2043
166,Kel'el Ware,Mark Williams,8.5,9.5,over,over,12.99,13.29,0.850,0.809,0.49,0.54,0.688,0.265,0.423,4.49,3.79,-102,-122,260,3.60,1.4756,147.56,0.1419
141,Anthony Black,Devin Booker,15.5,22.5,under,over,10.22,26.48,0.756,0.745,0.51,0.52,0.563,0.265,0.298,5.28,3.98,-110,-115,257,3.57,1.0109,101.09,0.0983
182,Dru Smith,Dillon Brooks,4.5,18.5,over,over,6.64,22.74,0.716,0.788,0.51,0.56,0.564,0.286,0.279,2.14,4.24,-110,-137,230,3.30,0.8624,86.24,0.0937
40,Jalen Suggs,Jalen Brunson,18.5,27.5,under,over,12.20,28.93,0.709,0.595,0.51,0.48,0.422,0.245,0.177,6.30,1.43,-110,105,291,3.91,0.6499,64.99,0.0558
70,Tyler Herro,Grayson Allen,22.5,15.5,over,over,24.51,17.27,0.642,0.646,0.49,0.52,0.415,0.255,0.160,2.01,1.77,100,-115,274,3.74,0.5504,55.04,0.0502
12,Paolo Banchero,Luguentz Dort,20.5,7.5,under,over,15.75,8.18,0.702,0.593,0.56,0.49,0.416,0.274,0.142,4.75,0.68,-136,-102,244,3.44,0.4326,43.26,0.0443
130,Davion Mitchell,Royce O'Neale,8.5,8.5,over,over,8.84,9.51,0.568,0.616,0.48,0.52,0.350,0.250,0.100,0.34,1.01,105,-114,285,3.85,0.3474,34.74,0.0305
265,Sandro Mamukelashvili,Alex Caruso,9.5,4.5,under,over,6.67,5.13,0.602,0.603,0.52,0.52,0.363,0.270,0.092,2.83,0.63,-115,-114,251,3.51,0.2727,27.27,0.0272
252,Mikal Bridges,Jalen Williams,15.5,18.5,over,over,16.03,18.83,0.564,0.556,0.49,0.50,0.314,0.245,0.069,0.53,0.33,-102,-105,287,3.87,0.2142,21.42,0.0187


## Top EVs for 2 leg bets

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2','PREDICTION 1', 'PREDICTION 2', 'MODEL_PROB 1', 'MODEL_PROB 2', 'SIDE 1', 'SIDE 2', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
underdogPairs.to_csv('data/props/ev_analysis/underdogPairs.csv', index=False)
underdogPairs

Computing predictions for 30 players...
[MIN] No data found for Wendell Carter Jr
Found 29 valid players
Generated 274 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,SIDE 1,SIDE 2,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
93,Jaime Jaquez Jr.,Isaiah Joe,10.5,7.5,100,-105,20.78,14.38,0.970,0.891,over,over,0.864,290,237.04,0.2043
166,Kel'el Ware,Mark Williams,8.5,9.5,-102,-122,12.99,13.29,0.850,0.809,over,over,0.688,260,147.56,0.1419
141,Anthony Black,Devin Booker,15.5,22.5,-110,-115,10.22,26.48,0.756,0.745,under,over,0.563,257,101.09,0.0983
182,Dru Smith,Dillon Brooks,4.5,18.5,-110,-137,6.64,22.74,0.716,0.788,over,over,0.564,230,86.24,0.0937
40,Jalen Suggs,Jalen Brunson,18.5,27.5,-110,105,12.20,28.93,0.709,0.595,under,over,0.422,291,64.99,0.0558
70,Tyler Herro,Grayson Allen,22.5,15.5,100,-115,24.51,17.27,0.642,0.646,over,over,0.415,274,55.04,0.0502
12,Paolo Banchero,Luguentz Dort,20.5,7.5,-136,-102,15.75,8.18,0.702,0.593,under,over,0.416,244,43.26,0.0443
130,Davion Mitchell,Royce O'Neale,8.5,8.5,105,-114,8.84,9.51,0.568,0.616,over,over,0.350,285,34.74,0.0305
265,Sandro Mamukelashvili,Alex Caruso,9.5,4.5,-115,-114,6.67,5.13,0.602,0.603,under,over,0.363,251,27.27,0.0272
252,Mikal Bridges,Jalen Williams,15.5,18.5,-102,-105,16.03,18.83,0.564,0.556,over,over,0.314,287,21.42,0.0187


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

prizepicksPairs = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2','PREDICTION 1', 'PREDICTION 2', 'MODEL_PROB 1', 'MODEL_PROB 2', 'SIDE 1', 'SIDE 2', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
prizepicksPairs.to_csv('data/props/ev_analysis/prizepicksPairs.csv', index=False)
prizepicksPairs

Computing predictions for 51 players...
[MIN] No data found for Wendell Carter Jr
Found 50 valid players
Generated 929 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,SIDE 1,SIDE 2,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
312,Jaime Jaquez Jr.,Isaiah Joe,10.0,7.5,-137,-105,20.78,14.38,0.978,0.891,over,over,0.871,238,194.44,0.2042
345,Kel'el Ware,Aaron Wiggins,8.5,9.0,-102,-137,12.99,15.88,0.850,0.881,over,over,0.749,243,156.98,0.1615
869,Mark Williams,Victor Wembanyama,9.5,20.5,-122,-109,13.29,24.97,0.809,0.779,over,over,0.630,249,119.79,0.1203
231,Anthony Black,Devin Booker,15.5,22.5,-110,-115,10.22,26.48,0.756,0.745,under,over,0.563,257,101.09,0.0983
376,Tristan da Silva,Dillon Brooks,8.5,19.0,-105,-137,10.97,22.74,0.712,0.760,over,over,0.541,238,82.91,0.0871
465,Dru Smith,Deandre Ayton,4.5,13.5,-110,-110,6.64,15.72,0.716,0.683,over,over,0.489,264,77.97,0.0738
155,Jalen Suggs,Ja'Kobe Walter,18.5,7.0,-110,-137,12.20,2.68,0.709,0.740,under,under,0.525,230,73.32,0.0797
502,Tyus Jones,Devin Vassell,4.0,13.5,-137,-104,0.87,15.52,0.754,0.653,under,over,0.492,239,66.74,0.0698
35,Desmond Bane,Rui Hachimura,22.0,11.5,-137,-102,16.32,12.84,0.729,0.629,under,over,0.459,243,57.47,0.0591
508,Jalen Brunson,Chet Holmgren,27.5,18.0,105,-137,28.93,13.51,0.595,0.706,over,under,0.420,255,49.05,0.0481


## 3 leg parlay

### Underdog picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') ]

underdogTrios = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'ODDS 1', 'ODDS 2', 'ODDS 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL_PROB 1', 'MODEL_PROB 2', 'MODEL_PROB 3', 'SIDE 1', 'SIDE 2', 'SIDE 3', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
underdogTrios.to_csv('data/props/ev_analysis/underdogTrios.csv', index=False)
underdogTrios.head()

Computing predictions for 30 players...
[MIN] No data found for Wendell Carter Jr
Found 29 valid players
Generated 840 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,ODDS 1,ODDS 2,ODDS 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_PROB 1,MODEL_PROB 2,MODEL_PROB 3,SIDE 1,SIDE 2,SIDE 3,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
370,Jaime Jaquez Jr.,Jalen Brunson,Isaiah Joe,10.5,27.5,7.5,100,105,-105,20.78,28.93,14.38,0.970,0.595,0.891,over,over,over,0.514,700,311.19,0.1111
751,Kel'el Ware,Sandro Mamukelashvili,Mark Williams,8.5,9.5,9.5,-102,-115,-122,12.99,6.67,13.29,0.850,0.602,0.809,over,under,over,0.414,574,178.88,0.0779
649,Anthony Black,Mikal Bridges,Devin Booker,15.5,15.5,22.5,-110,-102,-115,10.22,16.03,26.48,0.756,0.564,0.745,under,over,over,0.318,607,124.51,0.0513
796,Dru Smith,Brandon Ingram,Dillon Brooks,4.5,23.5,18.5,-110,-116,-137,6.64,20.65,22.74,0.716,0.587,0.788,over,under,over,0.331,515,103.79,0.0504
222,Jalen Suggs,Immanuel Quickley,Grayson Allen,18.5,17.5,15.5,-110,-104,-115,12.20,17.54,17.27,0.709,0.530,0.646,under,over,over,0.243,600,69.93,0.0291


### Prizepicks picks

In [10]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'ODDS 1', 'ODDS 2', 'ODDS 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL_PROB 1', 'MODEL_PROB 2', 'MODEL_PROB 3', 'SIDE 1', 'SIDE 2', 'SIDE 3', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
triosPrizepicks.to_csv('data/props/ev_analysis/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Computing predictions for 51 players...
[MIN] No data found for Wendell Carter Jr
Found 50 valid players
Generated 7588 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,ODDS 1,ODDS 2,ODDS 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_PROB 1,MODEL_PROB 2,MODEL_PROB 3,SIDE 1,SIDE 2,SIDE 3,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
3801,Jaime Jaquez Jr.,Isaiah Joe,Victor Wembanyama,10.0,7.5,20.5,-137,-105,-109,20.78,14.38,24.97,0.978,0.891,0.779,over,over,over,0.678,548,339.47,0.1549
4202,Kel'el Ware,Aaron Wiggins,Deandre Ayton,8.5,9.0,13.5,-102,-137,-110,12.99,15.88,15.72,0.850,0.881,0.683,over,over,over,0.511,554,234.47,0.1058
2798,Anthony Black,Ja'Kobe Walter,Mark Williams,15.5,7.0,9.5,-110,-137,-122,10.22,2.68,13.29,0.756,0.740,0.809,under,under,over,0.453,501,172.02,0.0858
4574,Tristan da Silva,Devin Booker,Devin Vassell,8.5,22.5,13.5,-105,-115,-104,10.97,26.48,15.52,0.712,0.745,0.653,over,over,over,0.346,616,147.96,0.0600
5434,Dru Smith,Dillon Brooks,Rui Hachimura,4.5,19.0,11.5,-110,-137,-102,6.64,22.74,12.84,0.716,0.760,0.629,over,over,over,0.343,554,124.05,0.0560
